# Data Analysis For All Weather Stations From IMS

In [2]:
import pandas as pd
from pathlib import Path
from weather_engine import utils as ut

def get_missingness_report(df: pd.DataFrame, file_path: Path):
    """
    Calculates missingness percentage for specific features and returns a summary row.
    """
    feature_cols = df.columns.tolist()
    
    station_name = Path(file_path).stem.split("_2020-2026")[0]

    if not df.empty:
        station_id = df['station_id'].iloc[0]
    else:
        station_id = 'Unknown'

    report = {
        'station_name': station_name,
        'station_id': station_id
    }
    
    for col in feature_cols:
        if df.empty:
            report[f"{col}_missingness"] = 100.0
        else:
            missing_pct = (df[col].isna().sum() / len(df)) * 100
            report[f"{col}_missingness"] = round(missing_pct, 2)
            
    return report


data_dir =  ut.get_project_root() / 'data/'
csv_files = list(data_dir.glob('*.csv'))

missingness_results = []
coords_list = []

print(f"Found {len(csv_files)} files. Starting cyclic processing...")

for file_path in csv_files:
    current_df = pd.read_csv(file_path)
    
    station_report = get_missingness_report(current_df, file_path)
    missingness_results.append(station_report)

    if not current_df.empty and 'latitude' in current_df.columns:
        first_valid = current_df[['station_id', 'latitude', 'longitude']].dropna().iloc[:1]
        first_valid['station_name'] = Path(file_path).stem.split("_2020-2026")[0]
        if not first_valid.empty:
            coords_list.append(first_valid.iloc[0].to_dict())
    
    del current_df

final_report_df = pd.DataFrame(missingness_results)
station_coords_df = pd.DataFrame(coords_list).astype({'station_id': str})


missingness_cols = [c for c in final_report_df.columns if c.endswith('_missingness')]

with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    styled_df = final_report_df.style.background_gradient(
        cmap='RdYlGn_r', 
        subset=missingness_cols, 
        vmin=0, 
        vmax=100
    )
    
    display(styled_df)

Found 89 files. Starting cyclic processing...


,station_name,station_id,timestamp_missingness,station_id_missingness,latitude_missingness,longitude_missingness,elevation_missingness,rain_missingness,wsmax_missingness,wdmax_missingness,ws_missingness,wd_missingness,stdwd_missingness,td_missingness,rh_missingness,tdmax_missingness,tdmin_missingness,ws1mm_missingness,ws10mm_missingness
0,MEROM_GOLAN_PICMAN,10,0.000000,0.000000,0.000000,0.000000,100.000000,0.140000,0.020000,0.020000,0.020000,0.030000,0.020000,0.020000,0.020000,0.020000,0.020000,0.020000,0.020000
1,KEFAR_BLUM,202,0.000000,0.000000,0.000000,0.000000,100.000000,9.410000,0.010000,0.050000,0.010000,0.050000,0.050000,0.020000,0.060000,0.030000,0.020000,0.010000,0.010000
2,NETIV_HALAMED_HE,25,0.000000,0.000000,0.000000,0.000000,100.000000,0.270000,0.370000,0.360000,0.370000,0.360000,0.340000,0.060000,0.030000,0.070000,0.100000,0.370000,0.370000
3,HAR_HARASHA,24,0.000000,0.000000,0.000000,0.000000,100.000000,1.780000,0.030000,0.090000,0.160000,0.090000,0.020000,0.020000,0.020000,0.020000,0.070000,0.020000,0.020000
4,GAT,236,0.000000,0.000000,0.000000,0.000000,100.000000,3.500000,100.000000,100.000000,100.000000,100.000000,100.000000,0.030000,0.050000,0.040000,0.020000,100.000000,100.000000
5,NEGBA,82,0.000000,0.000000,0.000000,0.000000,100.000000,0.000000,0.030000,0.380000,0.030000,0.380000,0.380000,0.050000,0.050000,0.030000,0.030000,0.030000,0.030000
6,DOROT,79,0.000000,0.000000,0.000000,0.000000,100.000000,0.890000,0.020000,0.030000,0.120000,0.030000,0.020000,0.050000,0.070000,0.050000,0.040000,0.020000,41.130000
7,EN_HAHORESH,107,0.000000,0.000000,0.000000,0.000000,100.000000,0.000000,100.000000,100.000000,100.000000,100.000000,100.000000,0.020000,0.020000,0.020000,0.020000,100.000000,100.000000
8,MIZPE_MIDRAG,504,0.000000,0.000000,0.000000,0.000000,100.000000,0.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
9,YOTVATA,36,0.000000,0.000000,0.000000,0.000000,100.000000,0.000000,0.020000,0.020000,0.020000,0.020000,0.020000,0.030000,0.020000,0.030000,0.030000,0.020000,17.780000


## Filtering out problematic stations that miss alot of sensors

In [3]:
cols_to_ignore = ['elevation_missingness', 
                    'ws1mm_missingness', 'ws10mm_missingness',
                    'timestamp_missingness', 'station_id_missingness', 
                    'latitude_missingness', 'longitude_missingness']

cols_to_check = [c for c in final_report_df.columns 
                 if c.endswith('_missingness') 
                 and c not in cols_to_ignore]

print(cols_to_check)

stations_to_leave = []

for idx, row in final_report_df.iterrows():
    for col in cols_to_check:
        if row[col] >= 30.0:
            stations_to_leave.append(row['station_id'])
            break

df_bad_stations = final_report_df[final_report_df['station_id'].isin(stations_to_leave)]
print(len(df_bad_stations))
df_bad_stations
            

['rain_missingness', 'wsmax_missingness', 'wdmax_missingness', 'ws_missingness', 'wd_missingness', 'stdwd_missingness', 'td_missingness', 'rh_missingness', 'tdmax_missingness', 'tdmin_missingness']
23


,station_name,station_id,timestamp_missingness,station_id_missingness,latitude_missingness,longitude_missingness,elevation_missingness,rain_missingness,wsmax_missingness,wdmax_missingness,ws_missingness,wd_missingness,stdwd_missingness,td_missingness,rh_missingness,tdmax_missingness,tdmin_missingness,ws1mm_missingness,ws10mm_missingness
4,GAT,236,0.0,0.0,0.0,0.0,100.0,3.50,100.00,100.00,100.00,100.00,100.00,0.03,0.05,0.04,0.02,100.00,100.00
7,EN_HAHORESH,107,0.0,0.0,0.0,0.0,100.0,0.00,100.00,100.00,100.00,100.00,100.00,0.02,0.02,0.02,0.02,100.00,100.00
8,MIZPE_MIDRAG,504,0.0,0.0,0.0,0.0,100.0,0.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
21,HAKFAR_HAYAROK,275,0.0,0.0,0.0,0.0,100.0,5.74,100.00,100.00,100.00,100.00,100.00,1.09,1.13,100.00,100.00,100.00,100.00
25,NEWE_ZOHAR_UNI,32,0.0,0.0,0.0,0.0,100.0,100.00,100.00,100.00,100.00,100.00,100.00,0.18,0.17,100.00,100.00,100.00,100.00
27,AMMIAD,123,0.0,0.0,0.0,0.0,100.0,0.00,100.00,100.00,100.00,100.00,100.00,1.93,1.93,1.93,1.93,100.00,100.00
30,BET_DAGAN_RAD,85,0.0,0.0,0.0,0.0,100.0,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
33,MASSADA,355,0.0,0.0,0.0,0.0,100.0,0.06,100.00,100.00,100.00,100.00,100.00,0.02,0.02,0.02,0.02,100.00,100.00
35,QEVUZAT_YAVNE,74,0.0,0.0,0.0,0.0,100.0,0.11,100.00,100.00,100.00,100.00,100.00,0.02,0.02,0.02,0.02,100.00,100.00
41,DEIR_HANNA,99,0.0,0.0,0.0,0.0,100.0,0.44,100.00,100.00,100.00,100.00,100.00,0.02,0.01,0.02,0.02,100.00,100.00


In [4]:
# How many stations have ANY column over 30%?
print(f"Total stations: {len(final_report_df)}")
print(f"Stations with 30%+ missingness in any column: {len(stations_to_leave)}")

# Which columns are causing the most flags?
for col in cols_to_check:
    over_30 = (final_report_df[col] >= 30.0).sum()
    if over_30 > 0:
        print(f"{col}: {over_30} stations over 30%")

Total stations: 89
Stations with 30%+ missingness in any column: 23
rain_missingness: 5 stations over 30%
wsmax_missingness: 22 stations over 30%
wdmax_missingness: 22 stations over 30%
ws_missingness: 22 stations over 30%
wd_missingness: 22 stations over 30%
stdwd_missingness: 22 stations over 30%
td_missingness: 4 stations over 30%
rh_missingness: 4 stations over 30%
tdmax_missingness: 6 stations over 30%
tdmin_missingness: 6 stations over 30%


In [5]:
df_survived_stations = final_report_df[~final_report_df['station_id'].isin(df_bad_stations['station_id'])]
print(len(df_survived_stations))

good_ids = df_survived_stations['station_id']
bad_ids = df_bad_stations['station_id']
shared = good_ids[good_ids.isin(bad_ids)]

if len(shared) > 0:
    print('Detected duplicated station in good and bad dataframes')
else:
    print('No duplicate stations were found in good and bad dataframes')

display(df_survived_stations.head())
display(df_bad_stations.head())

66
No duplicate stations were found in good and bad dataframes


,station_name,station_id,timestamp_missingness,station_id_missingness,latitude_missingness,longitude_missingness,elevation_missingness,rain_missingness,wsmax_missingness,wdmax_missingness,ws_missingness,wd_missingness,stdwd_missingness,td_missingness,rh_missingness,tdmax_missingness,tdmin_missingness,ws1mm_missingness,ws10mm_missingness
0,MEROM_GOLAN_PICMAN,10,0.0,0.0,0.0,0.0,100.0,0.14,0.02,0.02,0.02,0.03,0.02,0.02,0.02,0.02,0.02,0.02,0.02
1,KEFAR_BLUM,202,0.0,0.0,0.0,0.0,100.0,9.41,0.01,0.05,0.01,0.05,0.05,0.02,0.06,0.03,0.02,0.01,0.01
2,NETIV_HALAMED_HE,25,0.0,0.0,0.0,0.0,100.0,0.27,0.37,0.36,0.37,0.36,0.34,0.06,0.03,0.07,0.10,0.37,0.37
3,HAR_HARASHA,24,0.0,0.0,0.0,0.0,100.0,1.78,0.03,0.09,0.16,0.09,0.02,0.02,0.02,0.02,0.07,0.02,0.02
5,NEGBA,82,0.0,0.0,0.0,0.0,100.0,0.00,0.03,0.38,0.03,0.38,0.38,0.05,0.05,0.03,0.03,0.03,0.03


,station_name,station_id,timestamp_missingness,station_id_missingness,latitude_missingness,longitude_missingness,elevation_missingness,rain_missingness,wsmax_missingness,wdmax_missingness,ws_missingness,wd_missingness,stdwd_missingness,td_missingness,rh_missingness,tdmax_missingness,tdmin_missingness,ws1mm_missingness,ws10mm_missingness
4,GAT,236,0.0,0.0,0.0,0.0,100.0,3.50,100.0,100.0,100.0,100.0,100.0,0.03,0.05,0.04,0.02,100.0,100.0
7,EN_HAHORESH,107,0.0,0.0,0.0,0.0,100.0,0.00,100.0,100.0,100.0,100.0,100.0,0.02,0.02,0.02,0.02,100.0,100.0
8,MIZPE_MIDRAG,504,0.0,0.0,0.0,0.0,100.0,0.00,100.0,100.0,100.0,100.0,100.0,100.00,100.00,100.00,100.00,100.0,100.0
21,HAKFAR_HAYAROK,275,0.0,0.0,0.0,0.0,100.0,5.74,100.0,100.0,100.0,100.0,100.0,1.09,1.13,100.00,100.00,100.0,100.0
25,NEWE_ZOHAR_UNI,32,0.0,0.0,0.0,0.0,100.0,100.00,100.0,100.0,100.0,100.0,100.0,0.18,0.17,100.00,100.00,100.0,100.0


In [ ]:
import folium
import pandas as pd

coords = station_coords_df.copy()
coords['station_id'] = coords['station_id'].astype(float).astype(int).astype(str)

healthy_ids = set(df_survived_stations['station_id'].astype(str))
bad_ids = set(df_bad_stations['station_id'].astype(str))

healthy_coords = coords[coords['station_id'].isin(healthy_ids)]
bad_coords = coords[coords['station_id'].isin(bad_ids)]

m = folium.Map(tiles='cartodbdark_matter', location=[31.5, 34.8], zoom_start=8)

for idx, row in healthy_coords.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        color='green',
        fill=True,
        fill_opacity=0.8,
        popup=f"{row['station_name']} (ID: {row['station_id']})"
    ).add_to(m)

for idx, row in bad_coords.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        color='red',
        fill=True,
        fill_opacity=0.8,
        popup=f"{row['station_name']} (ID: {row['station_id']})"
    ).add_to(m)

m